In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import altair as alt
from matplotlib.transforms import ScaledTranslation

from scipy.stats import linregress, gaussian_kde

sys.path.append("../src")
from plot import create_scatter_correlation_plot
from enrichment_functions import create_customized_enrichment_plot

plt.style.use("../../config/DIT_HAP.mplstyle")
COLORS = plt.rcParams['axes.prop_cycle'].by_key()['color']
AX_WIDTH, AX_HEIGHT = plt.rcParams['figure.figsize']

plt.rcParams.update(
    {
        "font.family": "Times New Roman",
        "figure.titlesize": 30,
        "font.size": 24,
        "axes.titlesize": 28,
        "axes.labelsize": 26,
        "xtick.labelsize": 24,
        "ytick.labelsize": 24,
        "legend.fontsize": 22,
    }
)

output_dir = Path("../../results/HD_DIT_HAP_generationRAW/figures/thesis")
output_dir.mkdir(parents=True, exist_ok=True)

# 1. 代时计算

In [ ]:
time_vs_generations = pd.DataFrame(
    {
        "Samples": ["NSKY1328-4"] * 6 + ["NSKY1328-7"] * 6 + ["NSKY1328-8"] * 6,
        "Time (h)": [0,8,14,22,30,38,0,5,15.5,23,31,39,0,5,14.7,22.3,30.7,38.7],
        "Generations":[0,0,2.124,5.387,8.604,11.983,0,0,2.487,5.688,8.981,12.427,0,0,2.327,6.027,9.406,13.161]
    }
)

fig, ax = plt.subplots(figsize=(AX_WIDTH+2, AX_HEIGHT))

for sample, group in time_vs_generations.groupby("Samples"):
    ax.scatter(group["Time (h)"], group["Generations"], marker='o', label=sample, alpha=0.5)

# regression
x_values = time_vs_generations.query("Generations > 0")["Time (h)"]
y_values = time_vs_generations.query("Generations > 0")["Generations"]
slope, intercept, r_value, p_value, std_err = linregress(x_values, y_values)
x_fit = np.linspace(9, x_values.max(), 100)
slope = 0.422
intercept = -3.837
y_fit = slope * x_fit + intercept
ax.plot(x_fit, y_fit, color='darkred', ls="--")
ax.text(0.05, 0.95, f"PCC = {r_value:.3f}\nR² = {r_value**2:.3f}", transform=ax.transAxes, fontsize=20, verticalalignment='top')
ax.text(0.5, 0.3, f"y = {slope:.3f} * x {intercept:.3f}", transform=ax.transAxes, fontsize=20, verticalalignment='top', color='darkred')

ax.set_xlabel("Time (h)", fontsize=20)
ax.set_ylabel("Generations", fontsize=20)
ax.set_title("Time vs Generations for Different Samples", fontsize=22)
ax.legend(loc=(1.05,0.3), title="Samples", frameon=False)
plt.tight_layout()
plt.savefig(output_dir/"time_vs_generations.pdf", dpi=300, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
slope2, intercept2 = np.polyfit(x_values, y_values, 1)

# 2. PCR quality control

In [ ]:
LD_1328_7_PBL_PBR = pd.read_csv("../../results/LD_DIT_HAP_generationRAW/8_merged/LD1328-7_0h_YES.tsv", sep="\t", index_col=[0,1,2])

LD1328_7_NSK = pd.read_csv("../../results/LD_DIT_HAP_generationRAW/8_merged/LD1328-7_0h_YES.tsv", sep="\t", index_col=[0,1,2])
LD1328_7_YYS = pd.read_csv("../../results/Spore2YES6_1328/8_merged/LD1328-7_0h_YES.tsv", sep="\t", index_col=[0,1,2])
LD1328_7_merged = pd.merge(LD1328_7_NSK, LD1328_7_YYS, left_index=True, right_index=True, suffixes=("_NSK", "_YYS"))

LD1328_4_NSK = pd.read_csv("../../results/LD_DIT_HAP_generationRAW/8_merged/LD1328-4_0h_YES.tsv", sep="\t", index_col=[0,1,2])
LD1328_8_NSK = pd.read_csv("../../results/LD_DIT_HAP_generationRAW/8_merged/LD1328-8_0h_YES.tsv", sep="\t", index_col=[0,1,2])
LD1328_merged = pd.merge(LD1328_7_NSK, LD1328_8_NSK, left_index=True, right_index=True, suffixes=("_1", "_2"))

spike_in_data = pd.read_csv("../../results/Spikein/14_spikein_correlation/spike_in_results.tsv", sep="\t")
spike_in_data = spike_in_data.query("Sample != 'Spikein0'")

In [ ]:
fig, axes = plt.subplot_mosaic([["(a)", "(b)"], ["(c)", "(d)"]], figsize=(AX_WIDTH*2, (AX_HEIGHT)*2))

create_scatter_correlation_plot(
    LD_1328_7_PBL_PBR["PBL"].values,
    LD_1328_7_PBL_PBR["PBR"].values,
    ax=axes["(a)"],
    xscale="log",
    yscale="log"
)

create_scatter_correlation_plot(
    LD1328_7_merged["Reads_NSK"],
    LD1328_7_merged["Reads_YYS"],
    ax=axes["(b)"],
    xscale="log",
    yscale="log"
)

create_scatter_correlation_plot(
    LD1328_merged["Reads_1"],
    LD1328_merged["Reads_2"],
    ax=axes["(c)"],
    xscale="log",
    yscale="log"
)

ax = axes["(d)"]
for idx, (strain, strain_df) in enumerate(spike_in_data.groupby("Name")):

    X = strain_df["Relative_Dilution_Ratio"]
    Y = strain_df["Relative_Read_Ratio"]
    
    ax.scatter(X, Y, label=f"{strain}", facecolor="none", edgecolor=COLORS[idx], 
               s=150, lw=1.5, alpha=0.9)


slope, intercept, r_value, p_value, std_err = linregress(
    spike_in_data["Relative_Dilution_Ratio"],
    spike_in_data["Relative_Read_Ratio"]
)
r2_scores = r_value ** 2

line_x = np.array([-8, 0])
line_y = slope * line_x + intercept
ax.plot(line_x, line_y, color="black", ls="--", alpha=0.7, lw=2.5)

ax.set_xlabel("log$_{2}$(relative dilution ratio)")
ax.set_ylabel("log$_{2}$(relative read ratio)")
ax.set_xticks([-8, -6, -4, -2, 0])
ax.set_yticks([-8, -6, -4, -2, 0])
ax.set_xticklabels([-8, -6, -4, -2, 0])
ax.set_yticklabels([-8, -6, -4, -2, 0])

ax.text(0.05, 0.95, f"PCC={r_value:.2f}\nR²={r2_scores:.2f}\nSlope={slope:.2f}\nIntercept={intercept:.2f}", transform=ax.transAxes, ha="left", va="top")
ax.legend(loc="lower right", fontsize=16, frameon=False)

axes["(a)"].set_xlabel("PBL Reads")
axes["(a)"].set_ylabel("PBR Reads")

axes["(b)"].set_xlabel("Reads of Technical Replicate 1")
axes["(b)"].set_ylabel("Reads of Technical Replicate 2")

axes["(c)"].set_xlabel("Reads of Biological Replicate 1")
axes["(c)"].set_ylabel("Reads of Biological Replicate 2")
# for label, ax in axes.items():
#     ax.text(0, 1, label, transform=(ax.transAxes+ ScaledTranslation(-0.8, 0.8, fig.dpi_scale_trans)), fontsize=40, verticalalignment='top')

plt.tight_layout(h_pad=2, w_pad=2)
plt.savefig(output_dir/"PCR_quality_control.pdf", dpi=300, bbox_inches='tight')
plt.show()
plt.close()

# 3. Insertion distribution

In [ ]:
from matplotlib.ticker import FuncFormatter

fai_path = Path("/data/c/yangyusheng_optimized/DIT_HAP_pipeline/resources/pombase_data/2025-10-01/genome_sequence_and_features/Schizosaccharomyces_pombe_all_chromosomes.fa.fai")
chromosome_size = {}
with fai_path.open() as fh:
    for line in fh:
        parts = line.strip().split("\t")
        if len(parts) < 2:
            continue
        chrom_name = parts[0]
        try:
            chrom_len = int(parts[1])
        except ValueError:
            continue
        chromosome_size[chrom_name] = chrom_len

insertion_data = pd.read_csv("../../results/HD_DIT_HAP_generationRAW/15_insertion_level_curve_fitting/insertion_level_fitting_statistics.tsv", sep="\t")
plus_insertion = insertion_data.query("Strand == '+'").drop_duplicates(subset=["Chr", "Coordinate", "Strand", "Target"])
minus_insertion = insertion_data.query("Strand == '-'").drop_duplicates(subset=["Chr", "Coordinate", "Strand", "Target"])

# genome-wide insertion distribution per chromosome with widths proportional to chromosome lengths
chromosomes = ["I", "II", "III"]
window_size = 10000  # bin size in bp

# use width_ratios equal to chromosome lengths so subplot widths reflect real lengths
width_ratios = [
    ["I"] * chromosome_size["I"],
    ["II"] * chromosome_size["II"] + ["."]*(chromosome_size["I"] - chromosome_size["II"]),
    ["III"] * chromosome_size["III"] + ["."]*(chromosome_size["I"] - chromosome_size["III"])
]

# create a horizontal layout so widths are comparable; total figsize width can be tuned
fig, axes = plt.subplot_mosaic(width_ratios, figsize=(16,12), sharey=True)

for chrom in chromosomes:
    ax = axes[chrom]
    chrom_len = chromosome_size[chrom]
    edges = np.arange(0, chrom_len + window_size, window_size)
    mids = (edges[:-1] + edges[1:]) / 2.0

    plus_chr = plus_insertion.query("Chr == @chrom")["Coordinate"].values
    minus_chr = minus_insertion.query("Chr == @chrom")["Coordinate"].values

    plus_hist, _ = np.histogram(plus_chr, bins=edges)
    minus_hist, _ = np.histogram(minus_chr, bins=edges)

    ax.bar(mids / 1e6, plus_hist, width=window_size / 1e6 * 0.9, color=COLORS[0], align="center", label="Plus insertion")
    ax.bar(mids / 1e6, -minus_hist, width=window_size / 1e6 * 0.9, color=COLORS[1], align="center", label="Minus insertion")

    # max_hist = max(int(plus_hist.max()) if plus_hist.size else 0, int(minus_hist.max()) if minus_hist.size else 0)
    # ax.set_ylim(-max_hist * 1.15, max_hist * 1.15)
    ax.set_ylabel("Insertions")
    ax.set_title(f"Chromosome {chrom}")

    # X ticks in Mb
    mb_ticks = np.arange(0, chrom_len + 1, 1_000_000)
    ax.set_xticks(mb_ticks / 1e6)
    ax.set_xticklabels([str(int(x)) for x in mb_ticks / 1e6])
    ax.set_xlabel("Mb")
    ax.grid(axis="y", lw=0.3, alpha=0.5)
    # remove x-axis margin so plots span full chromosome length
    ax.set_xmargin(0)
    # ensure leftmost tick (0) is shown by explicitly setting x-limits to chromosome bounds (in Mb)
    ax.set_xlim(0, chrom_len / 1e6)
    # format y tick labels as positive numbers
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, pos: str(int(abs(y)))))
    ax.spines["right"].set_visible(True)
    ax.spines["top"].set_visible(True)
    

# common legend
handles, labels = axes[chromosomes[-1]].get_legend_handles_labels()
fig.legend(handles, labels, ncol=3, frameon=False, bbox_to_anchor=(0.6, 0.95))

plt.tight_layout()
plt.savefig(output_dir/"insertion_distribution_genomewide_proportional_widths.pdf", dpi=300, bbox_inches="tight")
plt.show()
plt.close()


# 4. Normalization

In [ ]:
res_deseq2 = pd.read_csv("../../results/HD_DIT_HAP_generationRAW/14_insertion_level_depletion_analysis/insertion_level_statistics.tsv", sep="\t", index_col=[0,1,2,3], header=[0,1])

In [ ]:
fig, axes = plt.subplots(1,4, figsize=(16, 5), sharex=True)

for idx, timepoint in enumerate(["YES1", "YES2", "YES3", "YES4"]):
    ax = axes[idx]
    data = res_deseq2[timepoint].query("baseMean < 5e5")
    x = data["baseMean"]
    y = data["log2FoldChange"]
    ax.scatter(x, y, alpha=0.1, s=10, edgecolor="gray", facecolor="none", rasterized=True)
    sig_data = data.query("padj < 0.05")
    x_sig = sig_data["baseMean"]
    y_sig = sig_data["log2FoldChange"]
    ax.scatter(x_sig, y_sig, alpha=0.3, s=10, color=COLORS[0], edgecolor="none", rasterized=True)
    ax.set_xscale("log")
    ax.set_ylim(-2.5, 10)
    ax.axhline(0, color=COLORS[1], ls="--", lw=1.5, alpha=0.5)
    if idx > 0:
        ax.spines["left"].set_visible(False)
        ax.set_yticks([])
        ax.set_yticklabels([])
        ax.set_yticks([], minor=True)
    else:
        ax.set_ylabel("log2 fold change")
    ax.set_xlabel("Base mean")
    ax.set_title(timepoint)
plt.tight_layout()
plt.savefig(output_dir/"insertion_level_MA_plot.pdf", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# 5. Plus insertion vs minus insertion

In [ ]:
insertion_annotations = pd.read_csv("../../results/HD_DIT_HAP_generationRAW/12_concatenated/annotations.tsv", sep="\t", index_col=[0,1,2,3])
imputation_statistics = pd.read_csv("../../results/HD_DIT_HAP_generationRAW/13_filtered/imputation_statistics.tsv", sep="\t", index_col=[0,1,2,3])
no_imputation_insertions = insertion_annotations[insertion_annotations.index.isin(imputation_statistics.query("num_of_imputed_insertions == 0").index)]
all_index_with_strand = no_imputation_insertions.index.droplevel("Strand")
in_gene_index_without_strand = no_imputation_insertions.query("Type != 'Intergenic region' and Distance_to_stop_codon > 4").index.droplevel("Strand")
intergenic_index_without_strand = all_index_with_strand.difference(in_gene_index_without_strand)
plus_vs_minus = res_deseq2["YES4"].unstack("Strand").dropna()

imputation_1_insertion = insertion_annotations[insertion_annotations.index.isin(imputation_statistics.query("num_of_imputed_insertions == 1").index)]
all_imputation1_with_strand = imputation_1_insertion.index.droplevel("Strand")
in_gene_imputation1_without_strand = imputation_1_insertion.query("Type != 'Intergenic region' and Distance_to_stop_codon > 4").index.droplevel("Strand")
intergenic_imputation1_without_strand = all_imputation1_with_strand.difference(in_gene_imputation1_without_strand)

imputation_2_insertion = insertion_annotations[insertion_annotations.index.isin(imputation_statistics.query("num_of_imputed_insertions == 2").index)]
all_imputation2_with_strand = imputation_2_insertion.index.droplevel("Strand")
in_gene_imputation2_without_strand = imputation_2_insertion.query("Type != 'Intergenic region' and Distance_to_stop_codon > 4").index.droplevel("Strand")
intergenic_imputation2_without_strand = all_imputation2_with_strand.difference(in_gene_imputation2_without_strand)

In [ ]:
fig, axes = plt.subplots(2,3, figsize=((AX_WIDTH-1)*3, AX_HEIGHT*2))

for col, col_filter in enumerate([all_index_with_strand, in_gene_index_without_strand, intergenic_index_without_strand]):
    for row, feature in enumerate(["baseMean", "log2FoldChange"]):
        ax = axes[row, col]
        data = plus_vs_minus.query("index in @col_filter")
        x = data[(feature, "+")]
        y = data[(feature, "-")]
        ax.scatter(x, y, alpha=0.1, s=10, edgecolor="gray", facecolor="none", rasterized=True)
        if feature == "baseMean":
            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.set_xlabel("Base mean (plus strand)")
            ax.set_ylabel("Base mean (minus strand)")
            # ax.set_title(f"Base mean: {['All insertions', 'In-gene insertions', 'Intergenic insertions'][col]}")
            ax.set_xlim(0.1, 1e6)
            ax.set_ylim(0.1, 1e6)
            log10_x = np.log10(x)
            log10_y = np.log10(y)
            slope, intercept, r_value, p_value, std_err = linregress(log10_x, log10_y)
            x_fit = np.linspace(-1, 6, 100)
            y_fit = slope * x_fit + intercept
            ax.plot(10**x_fit, 10**y_fit, color="darkred", ls="--", alpha=0.7)
            ax.text(0.05, 0.95, f"Data points: {len(x):,}\nPCC: {r_value:.2f}\nR²: {r_value**2:.2f}\nSlope: {slope:.2f}\nIntercept: {intercept:.2f}", transform=ax.transAxes, ha="left", va="top", color="darkred")
        else:
            ax.set_xlabel("log2 fold change (plus strand)")
            ax.set_ylabel("log2 fold change (minus strand)")
            # ax.set_title(f"log2 fold change: {['All insertions', 'In-gene insertions', 'Intergenic insertions'][col]}")
            ax.set_xlim(-2.5, 10)
            ax.set_ylim(-2.5, 10)
            slope, intercept, r_value, p_value, std_err = linregress(x, y)
            x_fit = np.linspace(-2.5, 10, 100)
            y_fit = slope * x_fit + intercept
            ax.plot(x_fit, y_fit, color="darkred", ls="--", alpha=0.7)
            ax.text(0.05, 0.95, f"Data points: {len(x):,}\nPCC: {r_value:.2f}\nR²: {r_value**2:.2f}\nSlope: {slope:.2f}\nIntercept: {intercept:.2f}", transform=ax.transAxes, ha="left", va="top", color="darkred")
plt.tight_layout(h_pad=5, w_pad=3)
plt.savefig(output_dir/"plus_vs_minus.pdf", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

In [ ]:
fig, axes = plt.subplots(2,3, figsize=((AX_WIDTH-1)*3, AX_HEIGHT*2))

for col, col_filter in enumerate([all_imputation1_with_strand, in_gene_imputation1_without_strand, intergenic_imputation1_without_strand]):
    for row, feature in enumerate(["baseMean", "log2FoldChange"]):
        ax = axes[row, col]
        data = plus_vs_minus.query("index in @col_filter")
        x = data[(feature, "+")]
        y = data[(feature, "-")]
        ax.scatter(x, y, alpha=0.1, s=10, edgecolor="gray", facecolor="none", rasterized=True)
        if feature == "baseMean":
            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.set_xlabel("Base mean (plus strand)")
            ax.set_ylabel("Base mean (minus strand)")
            ax.set_title(f"Base mean: {['All insertions', 'In-gene insertions', 'Intergenic insertions'][col]}")
            ax.set_xlim(0.1, 1e6)
            ax.set_ylim(0.1, 1e6)
            log10_x = np.log10(x)
            log10_y = np.log10(y)
            slope, intercept, r_value, p_value, std_err = linregress(log10_x, log10_y)
            x_fit = np.linspace(-1, 6, 100)
            y_fit = slope * x_fit + intercept
            # ax.plot(10**x_fit, 10**y_fit, color="darkred", ls="--", alpha=0.7)
            ax.text(0.05, 0.95, f"Data points: {len(x):,}\nPCC: {r_value:.2f}\nR²: {r_value**2:.2f}\nSlope: {slope:.2f}\nIntercept: {intercept:.2f}", transform=ax.transAxes, ha="left", va="top", color="darkred")
        else:
            ax.set_xlabel("log2 fold change (plus strand)")
            ax.set_ylabel("log2 fold change (minus strand)")
            ax.set_title(f"log2 fold change: {['All insertions', 'In-gene insertions', 'Intergenic insertions'][col]}")
            ax.set_xlim(-2.5, 10)
            ax.set_ylim(-2.5, 10)
            slope, intercept, r_value, p_value, std_err = linregress(x, y)
            x_fit = np.linspace(-2.5, 10, 100)
            y_fit = slope * x_fit + intercept
            # ax.plot(x_fit, y_fit, color="darkred", ls="--", alpha=0.7)
            ax.text(0.05, 0.95, f"Data points: {len(x):,}\nPCC: {r_value:.2f}\nR²: {r_value**2:.2f}\nSlope: {slope:.2f}\nIntercept: {intercept:.2f}", transform=ax.transAxes, ha="left", va="top", color="darkred")
plt.tight_layout()
# plt.savefig(output_dir/"plus_vs_minus.pdf", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

In [ ]:
fig, axes = plt.subplots(2,3, figsize=((AX_WIDTH-1)*3, AX_HEIGHT*2))

for col, col_filter in enumerate([all_imputation2_with_strand, in_gene_imputation2_without_strand, intergenic_imputation2_without_strand]):
    for row, feature in enumerate(["baseMean", "log2FoldChange"]):
        ax = axes[row, col]
        data = plus_vs_minus.query("index in @col_filter")
        x = data[(feature, "+")]
        y = data[(feature, "-")]
        ax.scatter(x, y, alpha=0.1, s=10, edgecolor="gray", facecolor="none", rasterized=True)
        if feature == "baseMean":
            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.set_xlabel("Base mean (plus strand)")
            ax.set_ylabel("Base mean (minus strand)")
            ax.set_title(f"Base mean: {['All insertions', 'In-gene insertions', 'Intergenic insertions'][col]}")
            ax.set_xlim(0.1, 1e6)
            ax.set_ylim(0.1, 1e6)
            log10_x = np.log10(x)
            log10_y = np.log10(y)
            slope, intercept, r_value, p_value, std_err = linregress(log10_x, log10_y)
            x_fit = np.linspace(-1, 6, 100)
            y_fit = slope * x_fit + intercept
            # ax.plot(10**x_fit, 10**y_fit, color="darkred", ls="--", alpha=0.7)
            ax.text(0.05, 0.95, f"Data points: {len(x):,}\nPCC: {r_value:.2f}\nR²: {r_value**2:.2f}\nSlope: {slope:.2f}\nIntercept: {intercept:.2f}", transform=ax.transAxes, ha="left", va="top", color="darkred")
        else:
            ax.set_xlabel("log2 fold change (plus strand)")
            ax.set_ylabel("log2 fold change (minus strand)")
            ax.set_title(f"log2 fold change: {['All insertions', 'In-gene insertions', 'Intergenic insertions'][col]}")
            ax.set_xlim(-2.5, 10)
            ax.set_ylim(-2.5, 10)
            slope, intercept, r_value, p_value, std_err = linregress(x, y)
            x_fit = np.linspace(-2.5, 10, 100)
            y_fit = slope * x_fit + intercept
            # ax.plot(x_fit, y_fit, color="darkred", ls="--", alpha=0.7)
            ax.text(0.05, 0.95, f"Data points: {len(x):,}\nPCC: {r_value:.2f}\nR²: {r_value**2:.2f}\nSlope: {slope:.2f}\nIntercept: {intercept:.2f}", transform=ax.transAxes, ha="left", va="top", color="darkred")
plt.tight_layout()
# plt.savefig(output_dir/"plus_vs_minus.pdf", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# 6. Genes with TTAA and Covered genes with TTAA

In [ ]:
import re
from Bio import SeqIO

fasta_path = Path("/data/c/yangyusheng_optimized/DIT_HAP_pipeline/resources/pombase_data/2025-10-01/genome_sequence_and_features/Schizosaccharomyces_pombe_all_chromosomes.fa")
bed_path = Path("/data/c/yangyusheng_optimized/DIT_HAP_pipeline/resources/pombase_data/2025-10-01/genome_region/coding_gene_primary_transcripts.bed")

# load genome sequences into memory as BioPython SeqRecord objects keyed by first token of header
seq_dict = SeqIO.to_dict(SeqIO.parse(fasta_path, "fasta"), key_function=lambda rec: rec.id.split()[0])

# read BED (expecting at least chrom,start,end; optional name/score/strand)
coding_gene_bed = pd.read_csv(bed_path, sep="\t", header=0)

# count TTAA
def count_ttaa(row, seq_dict):
    chrom = str(row["#Chr"])
    start = int(row["ParentalRegion_start"])
    end = int(row["ParentalRegion_end"])
    ttaa_count = seq_dict[chrom].seq[start:end].upper().count("TTAA")
    return ttaa_count
coding_gene_bed["TTAA_count"] = coding_gene_bed.apply(count_ttaa, axis=1, seq_dict=seq_dict)

genes_with_ttaa_count = coding_gene_bed[["#Chr", "Transcript", "Strand", "Systematic ID", "Type", "Name", "DeletionLibrary_essentiality", "ParentalRegion_start", "ParentalRegion_end", "ParentalRegion_length", "TTAA_count"]].drop_duplicates()
genes_with_insertion_count = pd.read_csv("../../reports/HD_DIT_HAP_generationRAW/insertion_density_analysis/insertion_density_analysis.tsv", sep="\t")
genes_with_count = pd.merge(genes_with_ttaa_count, genes_with_insertion_count[["Systematic ID", "total_insertions"]], on="Systematic ID", how="left").fillna({"total_insertions": 0})

In [ ]:
genes_with_count['Is_Covered'] = genes_with_count['total_insertions'] > 0
genes_with_count['TTAA_Group'] = genes_with_count['TTAA_count'].apply(lambda x: 'Has TTAA' if x > 0 else 'No TTAA')

# ==========================================
# 2. 绘图设置
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(AX_HEIGHT*2, AX_HEIGHT))
ax1, ax2 = axes

# 外圈: TTAA Group
group_counts = genes_with_count['TTAA_Group'].value_counts()
group_counts = group_counts.reindex(['Has TTAA', 'No TTAA'])

# 1. Has TTAA -> Covered vs Missed
n_has_cov = len(genes_with_count[(genes_with_count['TTAA_count'] > 0) & (genes_with_count['Is_Covered'])])
n_has_mis = len(genes_with_count[(genes_with_count['TTAA_count'] > 0) & (~genes_with_count['Is_Covered'])])
# 2. No TTAA -> Noise vs Clean
n_no_cov = len(genes_with_count[(genes_with_count['TTAA_count'] == 0) & (genes_with_count['Is_Covered'])])
n_no_cln = len(genes_with_count[(genes_with_count['TTAA_count'] == 0) & (~genes_with_count['Is_Covered'])])

sizes_outer = [group_counts['Has TTAA'], group_counts['No TTAA']]
sizes_inner = [n_has_cov, n_has_mis, n_no_cov, n_no_cln]

colors_outer = [COLORS[1], "lightgray"]
wedges1, texts1, autotexts1 = ax1.pie(
    sizes_outer,
    radius=1.2,
    colors=colors_outer,
    autopct=lambda pct: f'{pct:.1f}%\n({int(pct/100*sum(sizes_outer)):,})',
    pctdistance=0.85,
    wedgeprops=dict(width=0.4, edgecolor='w'),
    startangle=90,
    textprops={'fontsize': 16},
)

colors_inner = [COLORS[0], COLORS[2], COLORS[3], "lightgray"]
wedges2, texts2, autotexts2 = ax1.pie(
    sizes_inner,
    autopct=lambda pct: f'{pct:.1f}%\n({int(pct/100*sum(sizes_inner)):,})',
    pctdistance=0.7,
    radius=0.8,
    colors=colors_inner,
    wedgeprops=dict(width=0.4, edgecolor='w'),
    startangle=90,
    textprops={'fontsize': 10},
)

ax1.text(0, 0, f"Total Genes\nN={genes_with_count.shape[0]}", ha='center', va='center', fontweight='bold', fontsize=12)

# 添加自定义图例 (因为 Pie 图自动图例很乱)
# outer legend (TTAA presence)
patches_outer = [plt.Rectangle((0,0),1,1, color=c) for c in colors_outer]
labels_outer = ['Genes with TTAA', 'Genes without TTAA']
leg_outer = ax1.legend(patches_outer, labels_outer, loc='upper center', bbox_to_anchor=(0.3, -0.1), ncol=1, fontsize=10, frameon=True)

# inner legend (coverage categories)
patches_inner = [plt.Rectangle((0,0),1,1, color=c) for c in colors_inner]
labels_inner = ['Covered (Has TTAA)', 'Missed (Has TTAA)', 'Covered (No TTAA)', 'Clean (No TTAA)']
leg_inner = ax1.legend(patches_inner, labels_inner, loc='upper center', bbox_to_anchor=(0.7, -0.1), ncol=1, fontsize=10, frameon=True)

# ensure both legends are shown
ax1.add_artist(leg_outer)

x = genes_with_count['TTAA_count']+1
y = genes_with_count['total_insertions']+1
logx = np.log10(x)
logy = np.log10(y)
xy = np.vstack([logx, logy])
z = gaussian_kde(xy)(xy)
scatter = ax2.scatter(x, y, c=z, s=20, edgecolor='none', cmap='viridis', alpha=0.7)
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel('Number of TTAA sites per gene (+1)')
ax2.set_ylabel('Number of insertions per gene (+1)')

x_diagonal = np.linspace(1, 150, 100)
y_diagonal = x_diagonal
ax2.plot(x_diagonal, y_diagonal, color='red', ls='--', lw=1.5, label='y=x')


plt.tight_layout()
plt.savefig(output_dir/"genes_with_ttaa_and_coverage_pie_and_scatter.svg", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# 7. Depletion curve examples

In [ ]:
gene_level_data = pd.read_csv("../../results/HD_DIT_HAP_generationRAW/17_gene_level_curve_fitting/gene_level_fitting_statistics.tsv", sep="\t", index_col=[0])

In [ ]:
def sigmoid_function(x: np.ndarray, A: float, um: float, lam: float) -> np.ndarray:
    """Calculate sigmoid function values with numerical stability using gompertz function."""
    if A == 0:
        return np.zeros_like(x)
    alpha = (um * np.e) / A
    u = alpha * (lam - x) + 1
    exponent = np.clip(u, -700, 700)
    return A * np.exp(-np.exp(exponent))

def curve_plot(ax, gene, data, show_fitting=True):
    if gene in data['Name'].tolist():
        gene_data = data.query(f"Name == '{gene}'").iloc[0]
    elif gene in data.index.tolist():
        gene_data = data.loc[gene]
    else:
        raise ValueError(f"Gene '{gene}' not found in curve data.")
    x = [0.0, 2.352, 5.588, 9.104, 12.48]
    y = gene_data[["YES0", "YES1", "YES2", "YES3", "YES4"]].values
    A, um, lam = gene_data[["A", "um", "lam"]].values

    if show_fitting:
        ax.scatter(x, y, color=COLORS[1], alpha=0.8, edgecolors='white')        
        # Plot fitted curve
        x_smooth = np.linspace(min(x), max(x), 100)
        y_fit = sigmoid_function(x_smooth, A, um, lam)
        ax.plot(x_smooth, y_fit, color=COLORS[3], alpha=0.6)
        x_line = np.linspace(lam-1, 14, 100)
        y_line = um * (x_line - lam)
        ax.plot(x_line, y_line, color=COLORS[0], linestyle='--', alpha=0.6)
        ax.axvline(x=lam, color=COLORS[2], linestyle='--', alpha=0.8)
    else:
        ax.plot(x, y, marker='o', color=COLORS[1], alpha=0.8)

    # Set axis limits to include (0,0) and move spines to zero
    ax.set_xlim(-1, 13)
    ax.set_ylim(-1, 7)

    # Move spines to zero
    ax.spines['left'].set_position('zero')
    ax.spines['bottom'].set_position('zero')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.tick_params(axis="both", which="major", labelsize=22, labelleft=True, labelbottom=True, left=True, bottom=True)
    ax.tick_params(axis='both', which='minor', left=False, bottom=False)

    ax.set_xlabel("Generation", fontsize=24)
    ax.set_ylabel("LFC", fontsize=24)
    ax.set_title(gene, fontstyle='italic', fontweight='bold', fontsize=26)

    return ax

In [ ]:
['#dd8369', '#6b99df', '#98a64e', '#a78bd9', '#64af6d', '#d57fbd', '#4bb29c', '#e0788f', '#4aadce', '#c4954b']

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)
axes = axes.flatten()
labels = {
    "SPAC1002.16c": "tna1",
    "SPAC23C11.15": "pst2",
    "SPBC577.02": "rpl3801",
    "SPAC13G6.09": "tsr402",
    "SPAC1002.08c": "mtf1",
    "SPBC1773.10c": "nrs1",
}

for idx, (sysID, gene) in enumerate(labels.items()):
    ax = axes[idx]
    curve_plot(ax, gene=gene, data=gene_level_data, show_fitting=False)
    if idx % 3 != 0:
        ax.set_ylabel("")
    if idx < 3:
        ax.set_xlabel("")
plt.tight_layout()
plt.savefig(output_dir / "gene_examples_no_fitting.pdf", dpi=300, bbox_inches='tight', transparent=True)
plt.show()
plt.close()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

curve_plot(ax, "mtf1", gene_level_data)
A, um, lam = gene_level_data.query("Name == 'mtf1'")[["A", "um", "lam"]].values[0]
ax.axhline(A, color=COLORS[-1], linestyle='--', alpha=0.8)

plt.tight_layout()
plt.savefig(output_dir / "mtf1_curve_fitting.pdf", dpi=300, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)
axes = axes.flatten()
labels = {
    "SPAC1002.16c": "tna1",
    "SPAC23C11.15": "pst2",
    "SPBC577.02": "rpl3801",
    "SPAC13G6.09": "tsr402",
    "SPAC1002.08c": "mtf1",
    "SPBC1773.10c": "nrs1",
}

for idx, (sysID, gene) in enumerate(labels.items()):
    ax = axes[idx]
    curve_plot(ax, gene=gene, data=gene_level_data, show_fitting=True)
    if idx % 3 != 0:
        ax.set_ylabel("")
    if idx < 3:
        ax.set_xlabel("")
plt.tight_layout()
plt.savefig(output_dir / "gene_examples_fitting.pdf", dpi=300, bbox_inches='tight', transparent=True)
plt.show()
plt.close()

# 8. DR DL 分布特征

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(AX_WIDTH * 2, AX_HEIGHT * 2))

row_colors = ['#6b99df', '#dd8369', '#98a64e']
for col, col_feature in enumerate(["DR", "DL"]):
    if col_feature == "DR":
        feature_bins = np.arange(-0.2, 1.5, 0.05)
        xlims = (-0.2, 1.5)
        col_name = "um"
    else:
        feature_bins = np.arange(0, 15, 0.5)
        xlims = (0, 15)
        col_name = "lam"
    for row, row_essentiality in enumerate(["DeletionLibrary_essentiality.notna()", "DeletionLibrary_essentiality == 'E'", "DeletionLibrary_essentiality == 'V'"]):
        ax = axes[row, col]
        data = gene_level_data.query(row_essentiality)[col_name].dropna()
        ax.hist(data, bins=feature_bins, rwidth=0.9, color=row_colors[row])
        ax.set_xlim(xlims)
        ax.set_xlabel(col_feature, fontsize=26)
        ax.set_ylabel("Number of genes")
        # ax.set_title(f"{col_feature} distribution")
        # if row == 0:
        #     ax.set_title(f"{col_feature} distribution")
        # if col == 0:
        #     # ax.text(-0.1, 1.1, f"{['(a)', '(b)', '(c)'][row]}", transform=(ax.transAxes + ScaledTranslation(-0.6, 0.4, fig.dpi_scale_trans)), fontsize=28, verticalalignment='top')
        #     ax.set_title(f"{['All genes', 'Essential genes', 'Non-essential genes'][row]}", loc="right", fontsize=30)
        ax.tick_params(axis="both", which="major", labelsize=22)
plt.tight_layout(h_pad=2)
plt.savefig(output_dir / "gene_feature_distributions.pdf", dpi=300, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
g2t = lambda y: (y+3.933)/0.426

print(g2t(3.59))
print(g2t(5.04))
print(g2t(8.12))

# 9. Colony size

In [ ]:
colony_size_data = pd.read_excel("../../results/HD_DIT_HAP_generationRAW/20_essentiality_verification/organized_verification_summary.xlsx")

In [ ]:
WT2E = colony_size_data.query("Category == 'WT' and DR > 0.35")
WT2E["verification_phenotype"].value_counts()

In [ ]:
WT2E.query("verification_phenotype == 'E,small colonies'")

In [ ]:
from scipy.stats import t as t_dist

fig, axes = plt.subplots(2,3, figsize=(AX_WIDTH*3, AX_HEIGHT*2), sharex=True, sharey=True)

for col, col_category in enumerate(["very small colonies", "small colonies", "WT"]):
    for row, row_day in enumerate(["day3", "day6"]):
        ax = axes[row, col]
        data = WT2E.query(f"verification_phenotype == '{col_category}'")
        x = data[f"median_area_{row_day}"]
        y = data["DR"]
        x, y = x[x.notna() & y.notna()], y[x.notna() & y.notna()]
        ax.scatter(x, y, color=COLORS[col], alpha=0.7)
        ax.set_xlabel("Ratio of median area to WT", fontsize=26)
        ax.set_ylabel("DR", fontsize=26)
        ax.set_title(f"{col_category} at {row_day}\nn={len(data)}", fontsize=30)
        ax.tick_params(axis="both", which="major", labelsize=22, labelbottom=True, labelleft=True)
        ax.set_ylim(0, 2)
        if col == 1:
            slope, intercept, r_value, p_value, std_err = linregress(x, y)
            x_fit = np.linspace(0, 1.2, 100)
            y_fit = slope * x_fit + intercept

            # Confidence interval for the mean prediction (95%)
            n = len(x)
            x_mean = x.mean()
            Sxx = np.sum((x - x_mean) ** 2)
            residuals = y - (slope * x + intercept)
            s_err = np.sqrt(np.sum(residuals ** 2) / (n - 2))
            alpha = 0.05
            tval = t_dist.ppf(1 - alpha / 2, df=n - 2)
            se_fit = s_err * np.sqrt(1 / n + (x_fit - x_mean) ** 2 / Sxx)
            delta = tval * se_fit
            y_fit_upper = y_fit + delta
            y_fit_lower = y_fit - delta

            ax.plot(x_fit, y_fit, color="darkred", ls="--", alpha=0.7)
            ax.fill_between(x_fit, y_fit_lower, y_fit_upper, color="darkred", alpha=0.2)
            ax.text(
                0.05,
                0.95,
                f"PCC: {r_value:.2f}\nR²: {r_value**2:.2f}\nSlope: {slope:.2f}\nIntercept: {intercept:.2f}",
                transform=ax.transAxes,
                ha="left",
                va="top",
                color="darkred",
                fontsize=14,
            )
plt.tight_layout()
plt.savefig(output_dir / "WT2E_colony_size_vs_DR.pdf", dpi=300, bbox_inches='tight')
plt.show()
plt.close()

# 10. cluster merging

In [ ]:
cluster_data = pd.read_csv("../../results/HD_DIT_HAP_generationRAW/18_gene_level_clustering/kmeans_cluster_result.tsv", sep="\t")

In [ ]:
MULTI_COLORS = ["#452062",
"#8be93d",
"#6d2bde",
"#5bdb62",
"#cc3ce7",
"#cfe044",
"#5447d1",
"#e5c83b",
"#9643bf",
"#6cac2e",
"#d93eb4",
"#63e3a1",
"#49288c",
"#b8e181",
"#6474df",
"#da9b33",
"#c875d8",
"#55a95c",
"#dd3d86",
"#69decb",
"#e63923",
"#63c0df",
"#db632f",
"#6897da",
"#c1712e",
"#425594",
"#9ca13e",
"#913379",
"#477126",
"#de4867",
"#3c8b67",
"#cc403c",
"#afddd8",
"#271c45",
"#e4cc7f",
"#9379c4",
"#90782c",
"#d9a5df",
"#2b4826",
"#dc76ab",
"#d0e2aa",
"#321521",
"#e1d2bf",
"#182f3b",
"#d49968",
"#537d9a",
"#8a3e1f",
"#b6bbdc",
"#652524",
"#97bc8f",
"#9e2e4c",
"#649f9e",
"#612643",
"#788a63",
"#b15f6a",
"#385f5d",
"#e28c7e",
"#3f311f",
"#d4a4b1",
"#605320",
"#8f6c8b",
"#b09a7e",
"#52485f",
"#8a6653"]

cluster_colors = [
    "#d49968",
    "#dd8369",
    "#6b99df",
    "#98a64e",
    "#64af6d",
    "#a78bd9",
    "#d57fbd",
    "#c4954b",
    "#4bb29c",
    "#e0788f",
    "#4aadce",
]

color_set = [
    MULTI_COLORS, cluster_colors
]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(AX_WIDTH*2, AX_HEIGHT))

titles = [
    "Before merging clusters",
    "After merging clusters"
]

for col, col_cluster in enumerate(["cluster", "revised_cluster"]):
    ax = axes[col]
    for cluster, cluster_df in cluster_data.groupby(col_cluster, sort=True):
        cluster = int(cluster)
        x = cluster_df["um"]
        y = cluster_df["lam"]

        ax.scatter(x, y, c=color_set[col][cluster], s=20, label=f"Cluster {cluster}\n(n={len(x)})", rasterized=True)
        centroid_x = x.mean()
        centroid_y = y.mean()
        ax.text(centroid_x, centroid_y, f"{cluster}", fontweight="bold", fontsize=13 * (col+1),
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="none", alpha=0.8))


        ax.set_xlabel("DR", fontsize=22)
        ax.set_ylabel("DL", fontsize=22)
        # ax.set_xticks(np.arange(0, 2, 0.1))
        # ax.set_yticks(np.arange(0, 10, 1))
        ax.tick_params(axis='both', which='major', labelsize=20, labelleft=True, labelbottom=True)
        # ax.grid(True, linestyle="--", alpha=0.5, color="gray")
        ax.set_xlim(-0.15, 1.45)
        ax.set_title(titles[col], fontsize=30)

plt.tight_layout(w_pad=4)
plt.savefig(output_dir / "gene_clusters_in_DR_DL_space.pdf", dpi=300, bbox_inches='tight')
plt.show()
plt.close()

# 11. Enrichment Visualization

In [ ]:
slim_BP_res = pd.read_excel("../../results/HD_DIT_HAP_generationRAW/19_enrichment_analysis/gene_ontology_enrichment_results.xlsx", sheet_name="Slim BP")
slim_CC_res = pd.read_excel("../../results/HD_DIT_HAP_generationRAW/19_enrichment_analysis/gene_ontology_enrichment_results.xlsx", sheet_name="Slim CC")

In [ ]:
slim_BP_chart = create_customized_enrichment_plot(slim_BP_res, title="Slim BP Enrichment", x_col="Cluster", y_col="term", color_col="p_fdr", size_col="term_coverage")
slim_CC_chart = create_customized_enrichment_plot(slim_CC_res, title="Slim CC Enrichment", x_col="Cluster", y_col="term", color_col="p_fdr", size_col="term_coverage")

slim_charts = [slim_BP_chart, slim_CC_chart]
alt.hconcat(*slim_charts).save(output_dir / "slim_enrichment_charts.pdf")

# 12. Cytoplasmic translation